# DeltaGateNet (Colab)

Train [DeltaGateNet](https://arxiv.org/abs/2602.14071) on SEED-VIG or SADT from Google Colab.

This notebook is self-contained. Put the datasets on Drive using the same layout as the README, set `DATA_DIR` below, then **Runtime → Run all**.

In [ ]:
from google.colab import drive

try:
    drive.mount("/content/drive")
except Exception:
    print("Failed to mount drive!")


In [ ]:
# Dataset root on Google Drive (must match the README tree)
DATASET = "seed-vig"  # "seed-vig" or "sadt"
DATA_DIR = "/content/drive/My Drive/Driving Fatigue Project/Data/SEED-VIG"
NUM_CHANNELS = 17
NUM_CLASSES = 3
MODE = "intra"  # "intra" or "inter"
N_FOLDS = 5
OUTPUT_DIR = "/content/logs"

# SADT example:
# DATASET = "sadt"
# DATA_DIR = "/content/drive/My Drive/Driving Fatigue Project/Data/SADT-2022"
# NUM_CHANNELS = 30
# NUM_CLASSES = 2


In [ ]:
%pip install numpy scipy scikit-learn matplotlib seaborn tqdm torch


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class BidirectionalDelta(nn.Module):
    """
    Bidirectional first-order temporal differencing.
    Separates positive and negative changes.

    Input : (B, C, T)
    Output: (B, 2*C, T)
    """

    def __init__(self):
        super().__init__()

    def forward(self, x):
        delta = x[:, :, 1:] - x[:, :, :-1]
        delta = F.pad(delta, (1, 0))

        delta_pos = F.relu(delta)
        delta_neg = F.relu(-delta)

        return torch.cat([delta_pos, delta_neg], dim=1)


BirectionalDelta = BidirectionalDelta


class GatedTemporalConv(nn.Module):
    def __init__(self, input_channels):
        super().__init__()

        hidden_dims = 16
        num_layers = 2
        dropout = 0.5
        kernel_size = 7

        self.input_channels = input_channels
        self.hidden_dims = hidden_dims

        self.input_proj = nn.Conv1d(
            in_channels=input_channels,
            out_channels=input_channels * hidden_dims,
            kernel_size=1,
            groups=1,
        )

        self.layers = nn.ModuleList()
        for _ in range(num_layers):
            self.layers.append(
                nn.Sequential(
                    nn.Conv1d(
                        in_channels=input_channels * hidden_dims,
                        out_channels=input_channels * hidden_dims,
                        kernel_size=kernel_size,
                        padding=kernel_size // 2,
                        groups=1,
                    ),
                    nn.BatchNorm1d(input_channels * hidden_dims),
                    nn.GELU(),
                    nn.Conv1d(
                        in_channels=input_channels * hidden_dims,
                        out_channels=input_channels * hidden_dims,
                        kernel_size=1,
                    ),
                    nn.Dropout(dropout),
                )
            )

        self.norm = nn.LayerNorm(hidden_dims)

    def forward(self, x):
        """
        x: (B, C, T)
        return: (B, C, hidden_dims)
        """
        B, C, T = x.shape

        x = self.input_proj(x)

        for layer in self.layers:
            x = x + layer(x)

        x = x.view(B, C, self.hidden_dims, T)
        x = x.mean(dim=-1)
        x = self.norm(x)

        return x


class MLP(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(MLP, self).__init__()

        hidden_dims = 16
        dropout_rate = 0.5

        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dims),
            nn.BatchNorm1d(hidden_dims),
            nn.LeakyReLU(0.3),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dims, hidden_dims),
            nn.BatchNorm1d(hidden_dims),
            nn.LeakyReLU(0.3),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dims, hidden_dims),
            nn.BatchNorm1d(hidden_dims),
            nn.LeakyReLU(0.3),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_dims, num_classes),
        )

    def forward(self, input):
        return self.mlp(input)


class DeltaGateNet(nn.Module):
    def __init__(self, num_channels, num_classes):
        super(DeltaGateNet, self).__init__()

        hidden_dims = 16

        self.temporal_diff = BidirectionalDelta()
        self.conv = GatedTemporalConv(input_channels=2 * num_channels)
        self.mlp = MLP(
            input_dim=2 * num_channels * hidden_dims,
            num_classes=num_classes,
        )

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d, nn.Conv3d)):
            nn.init.kaiming_normal_(
                module.weight,
                mode="fan_in",
                nonlinearity="relu",
            )
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)
        elif isinstance(module, (nn.BatchNorm1d, nn.BatchNorm2d, nn.BatchNorm3d)):
            nn.init.constant_(module.weight, 1)
            nn.init.constant_(module.bias, 0)

    def forward(self, eeg):
        batch_size = eeg.size(0)
        eeg = self.temporal_diff(eeg)
        eeg_features = self.conv(eeg)
        eeg_flat = eeg_features.view(batch_size, -1)
        return self.mlp(eeg_flat)


In [ ]:
import os

import numpy as np
from scipy.io import loadmat

SEED_VIG_FILES = [
    "1_20151124_noon_2.mat",
    "2_20151106_noon.mat",
    "3_20151024_noon.mat",
    "4_20151105_noon.mat",
    "4_20151107_noon.mat",
    "5_20141108_noon.mat",
    "5_20151012_night.mat",
    "6_20151121_noon.mat",
    "7_20151015_night.mat",
    "8_20151022_noon.mat",
    "9_20151017_night.mat",
    "10_20151125_noon.mat",
    "11_20151024_night.mat",
    "12_20150928_noon.mat",
    "13_20150929_noon.mat",
    "14_20151014_night.mat",
    "15_20151126_night.mat",
    "16_20151128_night.mat",
    "17_20150925_noon.mat",
    "18_20150926_noon.mat",
    "19_20151114_noon.mat",
    "20_20151129_night.mat",
    "21_20151016_noon.mat",
]


def create_tri_class_labels(perclos_values):
    """
    Convert PERCLOS values to tri-class labels:
    [0, 0.35] -> 0 (alert)
    (0.35, 0.7] -> 1 (semi-fatigue)
    (0.7, 1] -> 2 (fatigue)
    """
    labels = np.zeros_like(perclos_values)
    labels[(perclos_values > 0.35) & (perclos_values <= 0.7)] = 1
    labels[perclos_values > 0.7] = 2
    return labels.astype(int)


def load_seed_vig(data_dir, num_channels):
    """
    Load SEED-VIG raw EEG and PERCLOS labels.

    Expects:
        data_dir/Raw_Data/*.mat
        data_dir/perclos_labels/*.mat
    """
    raw_dir = os.path.join(data_dir, "Raw_Data")
    perclos_dir = os.path.join(data_dir, "perclos_labels")

    if not os.path.isdir(raw_dir) or not os.path.isdir(perclos_dir):
        raise FileNotFoundError(
            "SEED-VIG data not found. Expected "
            f"{raw_dir} and {perclos_dir}. "
            "See README.md for the official download and directory layout."
        )

    all_eeg = []
    all_y = []
    subject_ids = []

    for idx, filename in enumerate(SEED_VIG_FILES):
        raw_path = os.path.join(raw_dir, filename)
        perclos_path = os.path.join(perclos_dir, filename)

        if not os.path.isfile(raw_path):
            raise FileNotFoundError(f"Missing raw EEG file: {raw_path}")
        if not os.path.isfile(perclos_path):
            raise FileNotFoundError(f"Missing PERCLOS file: {perclos_path}")

        raw = loadmat(raw_path)
        perclos = loadmat(perclos_path)

        raw_eeg = np.array(raw["EEG"]["data"][0][0]).transpose()
        perclos_values = np.array(perclos["perclos"], dtype=float).squeeze()
        n_segments = len(perclos_values)

        eeg_segments = np.array_split(raw_eeg, n_segments, axis=1)
        eeg_data = np.stack(eeg_segments, axis=0)
        y = create_tri_class_labels(perclos_values).squeeze()

        if eeg_data.shape[1] != num_channels:
            raise ValueError(
                f"{filename}: expected {num_channels} EEG channels, "
                f"got {eeg_data.shape[1]}"
            )

        all_eeg.append(eeg_data)
        all_y.append(y)
        subject_ids.extend([idx] * len(y))

    eeg = np.concatenate(all_eeg, axis=0)
    labels = np.concatenate(all_y, axis=0)
    subject_ids = np.array(subject_ids)

    return eeg, labels, subject_ids

import glob
import os

import numpy as np
from scipy.io import loadmat


def _find_mat_file(data_dir):
    if os.path.isfile(data_dir) and data_dir.endswith(".mat"):
        return data_dir

    mat_files = sorted(glob.glob(os.path.join(data_dir, "*.mat")))
    if not mat_files:
        raise FileNotFoundError(
            f"No .mat file found in {data_dir}. "
            "Download the SADT Figshare release and place it under datasets/. "
            "See README.md for the official links and directory layout."
        )
    return mat_files[0]


def load_sadt(data_dir, num_channels):
    """
    Load SADT from a single .mat containing EEGsample, subindex, and substate.

    Works for both the balanced (2022) and unbalanced (2952) releases.
    Returns EEG as (N, C, T).
    """
    mat_path = _find_mat_file(data_dir)
    mat = loadmat(mat_path)

    for key in ("EEGsample", "subindex", "substate"):
        if key not in mat:
            raise KeyError(
                f"{mat_path} is missing '{key}'. "
                "The SADT Figshare files must keep EEGsample, subindex, and substate."
            )

    eeg = np.array(mat["EEGsample"])
    if eeg.ndim != 3:
        raise ValueError(f"EEGsample must be 3-D (N, C, T), got shape {eeg.shape}")

    if eeg.shape[1] != num_channels and eeg.shape[2] == num_channels:
        eeg = np.transpose(eeg, (0, 2, 1))

    if eeg.shape[1] != num_channels:
        raise ValueError(
            f"Expected {num_channels} EEG channels, got shape {eeg.shape}"
        )

    labels = np.array(mat["substate"]).squeeze().astype(int)
    subject_ids = np.array(mat["subindex"]).squeeze().astype(int)

    if len(eeg) != len(labels) or len(eeg) != len(subject_ids):
        raise ValueError(
            "EEGsample, substate, and subindex must have the same number of samples"
        )

    return eeg, labels, subject_ids

import numpy as np


def split_indices(subject_ids, mode, fold, n_folds=5, val_ratio=0.2):
    """
    Intra-subject: split each subject's samples into n folds.
    Inter-subject: hold out entire subjects.

    Randomness uses the process-wide numpy RNG (seeded by train.train.set_seed).
    """
    indices = np.arange(len(subject_ids))

    if mode == "intra":
        print(f"Running {n_folds}-Fold Intra-Subject Evaluation - Fold {fold + 1}/{n_folds}")

        train_indices = []
        test_indices = []

        unique_subjects = np.unique(subject_ids)
        for subj in unique_subjects:
            subj_indices = indices[subject_ids == subj]
            np.random.shuffle(subj_indices)

            fold_size = len(subj_indices) // n_folds
            fold_start = fold * fold_size
            fold_end = (fold + 1) * fold_size if fold < n_folds - 1 else len(subj_indices)

            test_indices.extend(subj_indices[fold_start:fold_end])
            train_indices.extend(
                np.concatenate(
                    [
                        subj_indices[:fold_start],
                        subj_indices[fold_end:],
                    ]
                )
            )

        train_indices = np.array(train_indices)
        test_indices = np.array(test_indices)

        np.random.shuffle(train_indices)
        val_size = int(val_ratio * len(train_indices))
        val_indices = train_indices[:val_size]
        train_indices = train_indices[val_size:]

    elif mode == "inter":
        print(f"Running {n_folds}-Fold Inter-Subject Evaluation - Fold {fold + 1}/{n_folds}")

        unique_subjects = np.unique(subject_ids)
        np.random.shuffle(unique_subjects)

        fold_size = len(unique_subjects) // n_folds
        fold_start = fold * fold_size
        fold_end = (
            (fold + 1) * fold_size if fold < n_folds - 1 else len(unique_subjects)
        )

        test_subjects = unique_subjects[fold_start:fold_end]
        remaining_subjects = np.concatenate(
            [
                unique_subjects[:fold_start],
                unique_subjects[fold_end:],
            ]
        )

        np.random.shuffle(remaining_subjects)
        split_point = int(0.8 * len(remaining_subjects))
        train_subjects = remaining_subjects[:split_point]
        val_subjects = remaining_subjects[split_point:]

        train_indices = indices[np.isin(subject_ids, train_subjects)]
        val_indices = indices[np.isin(subject_ids, val_subjects)]
        test_indices = indices[np.isin(subject_ids, test_subjects)]

        print(f"Train subjects: {train_subjects}")
        print(f"Val subjects: {val_subjects}")
        print(f"Test subjects: {test_subjects}")
    else:
        raise ValueError(f"Unknown mode '{mode}'. Use 'intra' or 'inter'.")

    return train_indices, val_indices, test_indices

DATASET_DEFAULTS = {
    "seed-vig": {"num_channels": 17, "num_classes": 3},
    "sadt": {"num_channels": 30, "num_classes": 2},
}


In [ ]:
import os

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


def plot_training_history(train_losses, val_losses, val_accuracies, mode, fold, save_path):
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, len(train_losses) + 1), train_losses, label="Training Loss", linewidth=2)
    plt.plot(range(1, len(val_losses) + 1), val_losses, label="Validation Loss", linewidth=2)
    plt.xlabel("Epochs")
    plt.ylabel("Loss")
    plt.title(f"{mode.capitalize()}-Subject CV - Fold {fold + 1}: Loss")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.plot(
        range(1, len(val_accuracies) + 1),
        val_accuracies,
        label="Validation Accuracy",
        linewidth=2,
        color="green",
    )
    plt.xlabel("Epochs")
    plt.ylabel("Accuracy")
    plt.title(f"{mode.capitalize()}-Subject CV - Fold {fold + 1}: Accuracy")
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.savefig(save_path, dpi=150)
    plt.close()


def evaluate_model(model, test_loader, num_classes, save_path=None):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for batch_eeg, batch_y in test_loader:
            batch_eeg = batch_eeg.to(device)
            batch_y = batch_y.to(device)

            outputs = model(batch_eeg)
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch_y.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    class_names = [f"class_{i}" for i in range(num_classes)]
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(num_classes)))
    plt.figure(figsize=(10, 10))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names,
    )
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=150)
    plt.close()

    print("=== Classification Metrics ===")
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Test Precision: {precision:.4f}")
    print(f"Test Recall: {recall:.4f}")
    print(f"Test F1-Score: {f1:.4f}")

    return accuracy, precision, recall, f1


In [ ]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, TensorDataset


SEED = 2026
NUM_EPOCHS = 200
LEARNING_RATE = 1e-4
BATCH_SIZE = 32


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def load_dataset(dataset, data_dir, num_channels):
    if dataset == "seed-vig":
        return load_seed_vig(data_dir, num_channels)
    if dataset == "sadt":
        return load_sadt(data_dir, num_channels)
    raise ValueError(f"Unknown dataset '{dataset}'. Use 'seed-vig' or 'sadt'.")


def train_model(model, train_loader, val_loader, checkpoint_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    print(f"Using device: {device}")

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    train_losses = []
    val_losses = []
    val_accuracies = []
    best_val_loss = float("inf")

    os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

    for epoch in range(NUM_EPOCHS):
        model.train()
        running_loss = 0.0

        for batch_eeg, batch_y in train_loader:
            batch_eeg = batch_eeg.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_eeg)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_train_loss = running_loss / len(train_loader)
        train_losses.append(avg_train_loss)

        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch_eeg, batch_y in val_loader:
                batch_eeg = batch_eeg.to(device)
                batch_y = batch_y.to(device)

                outputs = model(batch_eeg)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                probs = F.softmax(outputs, dim=1)
                preds = torch.argmax(probs, dim=1)

                all_preds.extend(preds.cpu().numpy().flatten())
                all_labels.extend(batch_y.cpu().numpy().flatten())

        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)

        accuracy = accuracy_score(all_labels, all_preds)
        val_accuracies.append(accuracy)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)

        print(
            f"Epoch [{epoch + 1}/{NUM_EPOCHS}]: "
            f"Train Loss: {avg_train_loss:.4f}, "
            f"Val Loss: {avg_val_loss:.4f}, "
            f"Val Accuracy: {accuracy:.4f}"
        )

    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Loaded Best DeltaGateNet Model.")

    return train_losses, val_losses, val_accuracies


def main(dataset, data_dir, num_channels, num_classes, mode="intra", fold=0, output_dir="./logs"):
    set_seed(SEED)

    all_eeg, all_y, subject_ids = load_dataset(dataset, data_dir, num_channels)
    train_indices, val_indices, test_indices = split_indices(subject_ids, mode=mode, fold=fold)

    eeg_train = all_eeg[train_indices]
    y_train = all_y[train_indices]
    eeg_val = all_eeg[val_indices]
    y_val = all_y[val_indices]
    eeg_test = all_eeg[test_indices]
    y_test = all_y[test_indices]

    train_dataset = TensorDataset(
        torch.from_numpy(eeg_train).float(),
        torch.from_numpy(y_train).long(),
    )
    val_dataset = TensorDataset(
        torch.from_numpy(eeg_val).float(),
        torch.from_numpy(y_val).long(),
    )
    test_dataset = TensorDataset(
        torch.from_numpy(eeg_test).float(),
        torch.from_numpy(y_test).long(),
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    print(
        f"Train samples: {len(train_dataset)}, "
        f"Val samples: {len(val_dataset)}, "
        f"Test samples: {len(test_dataset)}"
    )
    print(f"Train class distribution: {np.bincount(y_train, minlength=num_classes)}")
    print(f"Val class distribution: {np.bincount(y_val, minlength=num_classes)}")
    print(f"Test class distribution: {np.bincount(y_test, minlength=num_classes)}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = DeltaGateNet(num_channels=num_channels, num_classes=num_classes).to(device)

    fold_dir = os.path.join(output_dir, dataset, mode, f"fold_{fold + 1}")
    checkpoint_path = os.path.join(fold_dir, "best_model.pth")

    train_losses, val_losses, val_accuracies = train_model(
        model, train_loader, val_loader, checkpoint_path
    )

    print(f"\nPlotting training history for Fold {fold + 1}...")
    plot_training_history(
        train_losses,
        val_losses,
        val_accuracies,
        mode=mode,
        fold=fold,
        save_path=os.path.join(fold_dir, "training_history.png"),
    )

    accuracy, precision, recall, f1 = evaluate_model(
        model,
        test_loader,
        num_classes=num_classes,
        save_path=os.path.join(fold_dir, "confusion_matrix.png"),
    )

    return accuracy, precision, recall, f1


def run_cross_validation(
    dataset,
    data_dir,
    num_channels,
    num_classes,
    mode="intra",
    n_folds=5,
    output_dir="./logs",
):
    set_seed(SEED)

    all_metrics = {
        "accuracy": [],
        "precision": [],
        "recall": [],
        "f1": [],
    }

    for fold in range(n_folds):
        print(f"\n{'=' * 60}")
        print(f"Fold {fold + 1}/{n_folds}")
        print("=" * 60)

        accuracy, precision, recall, f1 = main(
            dataset=dataset,
            data_dir=data_dir,
            num_channels=num_channels,
            num_classes=num_classes,
            mode=mode,
            fold=fold,
            output_dir=output_dir,
        )

        all_metrics["accuracy"].append(accuracy)
        all_metrics["precision"].append(precision)
        all_metrics["recall"].append(recall)
        all_metrics["f1"].append(f1)

    print(f"\n{'=' * 60}")
    print(f"{mode.capitalize()}-Subject {n_folds}-Fold Cross Validation Results")
    print("=" * 60)

    for metric_name, values in all_metrics.items():
        mean_val = np.mean(values)
        std_val = np.std(values)
        print(f"Average {metric_name}: {mean_val:.4f} ± {std_val:.4f}")
        print(f"Individual fold {metric_name}s: {[f'{v:.4f}' for v in values]}")

    return all_metrics


In [ ]:
run_cross_validation(
    dataset=DATASET,
    data_dir=DATA_DIR,
    num_channels=NUM_CHANNELS,
    num_classes=NUM_CLASSES,
    mode=MODE,
    n_folds=N_FOLDS,
    output_dir=OUTPUT_DIR,
)
